In [2]:
import re

# All multi-char espeak phonemes (longest first — ORDER MATTERS)
ESPEAK_MULTI = [
    "a:", "i:", "u:", "O:", "3:", "e:",      # long vowels
    "eI", "oU", "aI", "aU", "OI",           # diphthongs
    "tS", "dZ",                              # affricates
    "t[", "d[",                              # dental stops
    "aa",                                    # mid-word schwa (our custom)
]

def tokenize_espeak(phoneme_str):
    """
    Split an espeak phoneme string into a list of phoneme tokens.
    Strips stress markers (' and ,) and keeps the phonemes.
    
    Example:
        "'a:dZun" → ['a:', 'dZ', 'u', 'n']
        "pr'aatSi:" → ['p', 'r', 'aa', 'tS', 'i:']
    """
    tokens = []
    i = 0
    # Remove stress marks first — they are not phonemes
    s = phoneme_str.replace("'", "").replace(",", "").replace("%", "")
    
    while i < len(s):
        matched = False
        # Try longest multi-char phoneme first
        for multi in ESPEAK_MULTI:
            if s[i:].startswith(multi):
                tokens.append(multi)
                i += len(multi)
                matched = True
                break
        if not matched:
            # Single character phoneme
            tokens.append(s[i])
            i += 1
    return tokens

# Test
print(tokenize_espeak("'a:dZun"))    # ['a:', 'dZ', 'u', 'n']
print(tokenize_espeak("Sr'i:ni:vas")) # ['S', 'r', 'i:', 'n', 'i:', 'v', 'a', 's']


['a:', 'dZ', 'u', 'n']
['S', 'r', 'i:', 'n', 'i:', 'v', 'a', 's']


In [3]:
# ALIGNMENT LOOKUP TABLE — Indian English specific
# Order is CRITICAL: longest patterns MUST come first
# (greedy left-to-right matching)

GRAPHEME_PRIORITY = [
    # === 3-CHAR GRAPHEMES (check first) ===
    "ksh",   # ksha → tSS (Lakshmi, Daksha)  
    "tth",   # double dental (Siddharth variants)
    "ddh",   # voiced double dental
    "ngh",   # nasal + gh cluster
    "shr",   # shri → Sr (Shrinivas, Shriram)
    "jnh",   # jnha
    
    # === 2-CHAR GRAPHEMES ===
    "sh",    # sh → S  (Shiva, Harish, Ashok)
    "th",    # th → t[ (Karthik, Rithika, Neethi)
    "dh",    # dh → d[ (Dharma, Madhav, Sindhu)
    "kh",    # kh → x  (Khan, Lakhani)
    "gh",    # gh → g  (Ghosh, Raghav)
    "bh",    # bh → b  (Bharat, Abhishek)
    "ph",    # ph → f  (Phalguni) or p (Deepak variant)
    "jh",    # jh → dZ (Jhoom, Jhansi)
    "ch",    # ch → tS (Chandra, Ruchi)
    "ng",    # ng → N  (Anga, Ranga)
    "ny",    # ny → nj (Anya, Pranya)
    "gy",    # gy → dZj (Gyan, Vigyan)
    "ck",    # ck → k  (Blackwood [rare])
    "aa",    # aa → a: (Raaman, Praan)
    "ee",    # ee → i: (Neelavathi, Preethi)
    "oo",    # oo → u: (Poonam, Goonjan)
    "ai",    # ai → eI or aI (Aishwarya, Vaibhav)
    "ou",    # ou → aU (Gourish)
    "rr",    # rr → r  (Gaur, Barr)
    "tt",    # tt → t  (Vittagen)
    "dd",    # dd → d  (Siddharth)
    "nn",    # nn → n  (Annamalai)
    "ll",    # ll → l  (Mallikarjun)
    "ss",    # ss → s  (Prasad)
    
    # === SINGLE CHARACTERS (fallback) ===
    "a", "b", "c", "d", "e", "f", "g", "h", "i",
    "j", "k", "l", "m", "n", "o", "p", "q", "r",
    "s", "t", "u", "v", "w", "x", "y", "z",
]

# Build the lookup: grapheme → most likely phoneme
# This is your STARTING POINT — will be refined by the data
GRAPHEME_TO_PHONEME_DEFAULT = {
    # 3-char
    "ksh": "tSS",  "shr": "Sr",  "jnh": "ndZ",  "ngh": "Ng",
    "tth": "t[",   "ddh": "d[",
    
    # 2-char
    "sh": "S",    "th": "t[",  "dh": "d[",  "kh": "x",
    "gh": "g",    "bh": "b",   "ph": "f",   "jh": "dZ",
    "ch": "tS",   "ng": "N",   "ny": "nj",  "gy": "dZj",
    "ck": "k",
    "aa": "a:",   "ee": "i:",  "oo": "u:",  
    "ai": "eI",   "ou": "aU",  "rr": "r",
    "tt": "t",    "dd": "d",   "nn": "n",
    "ll": "l",    "ss": "s",
    
    # Single vowels (context-dependent — will be learned from data)
    "a": "a",   "e": "E",   "i": "I",   "o": "O",   "u": "u",
    
    # Single consonants
    "b": "b",   "c": "k",   "d": "d",   "f": "f",   "g": "g",
    "h": "h",   "j": "dZ",  "k": "k",   "l": "l",   "m": "m",
    "n": "n",   "p": "p",   "q": "k",   "r": "r",   "s": "s",
    "t": "t",   "v": "v",   "w": "v",   "x": "ks",  "y": "j",
    "z": "z",
}


def scan_graphemes(word):
    """
    Scan a word left-to-right, greedily matching longest grapheme first.
    Returns a list of graphemes in order.
    
    Example:
        "shrinivas" → ['sh', 'r', 'i', 'n', 'i', 'v', 'a', 's']
        "karthik"   → ['k', 'a', 'r', 'th', 'i', 'k']
        "lakshmi"   → ['l', 'a', 'ksh', 'm', 'i']
    """
    graphemes = []
    i = 0
    word_lower = word.lower()
    while i < len(word_lower):
        matched = False
        # Try longest grapheme first
        for g in GRAPHEME_PRIORITY:
            if word_lower[i:].startswith(g):
                graphemes.append(g)
                i += len(g)
                matched = True
                break
        if not matched:
            graphemes.append(word_lower[i])
            i += 1
    return graphemes

# Test
print(scan_graphemes("shrinivas"))  # ['sh', 'r', 'i', 'n', 'i', 'v', 'a', 's']
print(scan_graphemes("karthik"))    # ['k', 'a', 'r', 'th', 'i', 'k']
print(scan_graphemes("lakshmi"))    # ['l', 'a', 'ksh', 'm', 'i']


['shr', 'i', 'n', 'i', 'v', 'a', 's']
['k', 'a', 'r', 'th', 'i', 'k']
['l', 'a', 'ksh', 'm', 'i']


In [4]:
def align(word, phoneme_str):
    """
    Align grapheme list to phoneme token list.
    Returns list of (grapheme, phoneme, left_ctx, right_ctx, position) tuples.
    
    Strategy: greedy left-to-right.
    When grapheme count == phoneme count → 1:1 alignment.
    When mismatch → use default lookup to resolve.
    """
    graphemes = scan_graphemes(word)
    phonemes  = tokenize_espeak(phoneme_str)
    
    aligned = []
    
    g_idx = 0  # grapheme pointer
    p_idx = 0  # phoneme pointer
    n_g   = len(graphemes)
    n_p   = len(phonemes)
    
    while g_idx < n_g and p_idx < n_p:
        g = graphemes[g_idx]
        
        # How many phonemes does our default say this grapheme produces?
        default_ph = GRAPHEME_TO_PHONEME_DEFAULT.get(g, g)
        default_count = 1 if isinstance(default_ph, str) else len(default_ph)
        
        # Take that many phonemes from actual sequence
        actual_ph = phonemes[p_idx : p_idx + default_count]
        actual_ph_str = "".join(actual_ph)
        
        # Position encoding
        if g_idx == 0:
            pos = "INITIAL"
        elif g_idx == n_g - 1:
            pos = "FINAL"
        elif g_idx == n_g - 2:
            pos = "PREFINAL"
        else:
            pos = "MIDDLE"
        
        # Context (what's before and after)
        left_1  = graphemes[g_idx - 1] if g_idx > 0     else "_"
        left_2  = graphemes[g_idx - 2] if g_idx > 1     else "_"
        right_1 = graphemes[g_idx + 1] if g_idx < n_g-1 else "_"
        right_2 = graphemes[g_idx + 2] if g_idx < n_g-2 else "_"
        
        aligned.append({
            "word":     word,
            "grapheme": g,
            "phoneme":  actual_ph_str,
            "left_2":   left_2,
            "left_1":   left_1,
            "right_1":  right_1,
            "right_2":  right_2,
            "position": pos,
        })
        
        g_idx += 1
        p_idx += default_count
    
    return aligned

# Test
for row in align("shrinivas", "Sr'i:ni:vas"):
    print(row)


{'word': 'shrinivas', 'grapheme': 'shr', 'phoneme': 'S', 'left_2': '_', 'left_1': '_', 'right_1': 'i', 'right_2': 'n', 'position': 'INITIAL'}
{'word': 'shrinivas', 'grapheme': 'i', 'phoneme': 'r', 'left_2': '_', 'left_1': 'shr', 'right_1': 'n', 'right_2': 'i', 'position': 'MIDDLE'}
{'word': 'shrinivas', 'grapheme': 'n', 'phoneme': 'i:', 'left_2': 'shr', 'left_1': 'i', 'right_1': 'i', 'right_2': 'v', 'position': 'MIDDLE'}
{'word': 'shrinivas', 'grapheme': 'i', 'phoneme': 'n', 'left_2': 'i', 'left_1': 'n', 'right_1': 'v', 'right_2': 'a', 'position': 'MIDDLE'}
{'word': 'shrinivas', 'grapheme': 'v', 'phoneme': 'i:', 'left_2': 'n', 'left_1': 'i', 'right_1': 'a', 'right_2': 's', 'position': 'MIDDLE'}
{'word': 'shrinivas', 'grapheme': 'a', 'phoneme': 'v', 'left_2': 'i', 'left_1': 'v', 'right_1': 's', 'right_2': '_', 'position': 'PREFINAL'}
{'word': 'shrinivas', 'grapheme': 's', 'phoneme': 'a', 'left_2': 'v', 'left_1': 'a', 'right_1': '_', 'right_2': '_', 'position': 'FINAL'}


In [5]:
from collections import Counter, defaultdict

def extract_rules(rutwik_extra_path, min_confidence=0.80, min_support=50):
    """
    Read rutwik_extra, align all words, count patterns,
    and output rules sorted by confidence × coverage.
    
    Returns a list of dicts:
        {pattern, phoneme, confidence, support, espeak_rule}
    """
    # STEP 1: Collect all aligned pairs
    all_rows = []
    
    with open(rutwik_extra_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("//"):
                continue
            parts = line.split("\t")
            if len(parts) != 2:
                continue
            word, phoneme_str = parts
            try:
                rows = align(word, phoneme_str)
                all_rows.extend(rows)
            except Exception:
                continue   # skip problem words
    
    print(f"Total aligned pairs collected: {len(all_rows)}")
    
    # STEP 2: Count pattern → phoneme frequencies
    # Pattern = (grapheme, left_1, right_1, position)
    # You can make it more or less specific by changing which fields you include
    
    pattern_counts = defaultdict(Counter)
    
    for row in all_rows:
        # Pattern 1: grapheme alone (most general)
        p1 = (row["grapheme"],)
        pattern_counts[p1][row["phoneme"]] += 1
        
        # Pattern 2: grapheme + left context
        p2 = (row["grapheme"], "L:" + row["left_1"])
        pattern_counts[p2][row["phoneme"]] += 1
        
        # Pattern 3: grapheme + right context
        p3 = (row["grapheme"], "R:" + row["right_1"])
        pattern_counts[p3][row["phoneme"]] += 1
        
        # Pattern 4: grapheme + position
        p4 = (row["grapheme"], "POS:" + row["position"])
        pattern_counts[p4][row["phoneme"]] += 1
        
        # Pattern 5: grapheme + left + right (most specific)
        p5 = (row["grapheme"], "L:" + row["left_1"], "R:" + row["right_1"])
        pattern_counts[p5][row["phoneme"]] += 1
    
    # STEP 3: Find reliable rules
    rules = []
    
    for pattern, ph_counter in pattern_counts.items():
        total = sum(ph_counter.values())
        best_ph, best_count = ph_counter.most_common(1)[0]
        confidence = best_count / total
        
        # Filter by confidence AND minimum support
        if confidence >= min_confidence and total >= min_support:
            rules.append({
                "pattern":    pattern,
                "phoneme":    best_ph,
                "confidence": round(confidence, 3),
                "support":    total,
                "coverage":   best_count,
            })
    
    # STEP 4: Sort by coverage (most words covered first)
    rules.sort(key=lambda r: r["coverage"], reverse=True)
    
    return rules


def pattern_to_espeak(rule):
    """
    Convert a mined rule dict to eSpeak rule syntax string.
    """
    pattern = rule["pattern"]
    grapheme = pattern[0]
    phoneme  = rule["phoneme"]
    conf     = rule["confidence"]
    sup      = rule["support"]
    
    # Build context qualifiers
    left_ctx  = ""
    right_ctx = ""
    pos_mod   = ""
    
    for part in pattern[1:]:
        if part.startswith("L:"):
            left_ctx = part[2:] + ") "
        elif part.startswith("R:"):
            right_ctx = " (" + part[2:]
        elif part == "POS:FINAL":
            right_ctx = " (_"
        elif part == "POS:INITIAL":
            left_ctx = "_) "
    
    rule_str = f"  {left_ctx}{grapheme}{right_ctx}   {phoneme}   $p1"
    comment  = f"  // conf={conf:.0%} n={sup}"
    
    return rule_str + comment


# === RUN IT ===
rules = extract_rules("/Users/rutwik/espeak-ng/dictsource/rutwik_extra")

print("\n=== TOP 30 RULES BY COVERAGE ===")
print(f"{'PATTERN':<35} {'PHONEME':<10} {'CONF':>8} {'N':>8}")
print("─" * 65)
for r in rules[:30]:
    pat_str = " + ".join(r["pattern"])
    print(f"{pat_str:<35} {r['phoneme']:<10} {r['confidence']:>7.1%} {r['support']:>8,}")

# Write rules to file
print("\n=== ESPEAK RULE FORMAT ===")
for r in rules[:30]:
    print(pattern_to_espeak(r))


Total aligned pairs collected: 767773

=== TOP 30 RULES BY COVERAGE ===
PATTERN                             PHONEME        CONF        N
─────────────────────────────────────────────────────────────────
n                                   n            80.6%   61,606
n + L:a                             n            80.6%   31,798
m                                   m            82.8%   28,909
a + R:_                             a            83.4%   24,749
a + POS:FINAL                       a            83.4%   24,749
v                                   v            84.1%   24,523
l                                   l            81.6%   24,213
d                                   d            83.8%   17,950
n + R:_                             n            85.3%   15,197
n + POS:FINAL                       n            85.3%   15,197
p                                   p            86.1%   14,268
l + POS:MIDDLE                      l            80.2%   15,239
j                            

In [6]:
# What base English already does by default (no rule needed)
BASE_ENGLISH_DEFAULT = {
    "n": "n",  "m": "m",  "l": "l",  "p": "p",
    "b": "b",  "d": "d",  "k": "k",  "g": "g",
    "f": "f",  "s": "s",  "z": "z",  "h": "h",
    "r": "r",  "v": "v",  "w": "w",  "j": "j",  # NOTE: j is tricky
}

def is_useful_rule(rule):
    pattern  = rule["pattern"]
    grapheme = pattern[0]
    phoneme  = rule["phoneme"]
    
    # Skip if this is just confirming the base English default
    base_default = BASE_ENGLISH_DEFAULT.get(grapheme)
    if base_default == phoneme:
        return False
    
    # Skip duplicates (deduplicate by espeak rule string)
    return True

# Also deduplicate by the espeak rule string itself
seen_rules = set()

useful_rules = []
for r in rules:
    espeak_str = pattern_to_espeak(r)
    if is_useful_rule(r) and espeak_str not in seen_rules:
        seen_rules.add(espeak_str)
        useful_rules.append(r)


In [8]:
useful_rules

[{'pattern': ('a', 'R:_'),
  'phoneme': 'a',
  'confidence': 0.834,
  'support': 24749,
  'coverage': 20652},
 {'pattern': ('j',),
  'phoneme': 'dZ',
  'confidence': 0.851,
  'support': 13573,
  'coverage': 11544},
 {'pattern': ('i', 'R:_'),
  'phoneme': 'i:',
  'confidence': 0.834,
  'support': 10395,
  'coverage': 8669},
 {'pattern': ('t', 'R:a'),
  'phoneme': 't',
  'confidence': 0.823,
  'support': 7128,
  'coverage': 5865},
 {'pattern': ('j', 'L:_'),
  'phoneme': 'dZ',
  'confidence': 0.975,
  'support': 5650,
  'coverage': 5506},
 {'pattern': ('j', 'R:a'),
  'phoneme': 'dZ',
  'confidence': 0.845,
  'support': 6179,
  'coverage': 5223},
 {'pattern': ('sh', 'L:i'),
  'phoneme': 'S',
  'confidence': 0.873,
  'support': 4317,
  'coverage': 3769},
 {'pattern': ('i', 'R:sh'),
  'phoneme': 'I',
  'confidence': 0.821,
  'support': 4446,
  'coverage': 3650},
 {'pattern': ('bh',),
  'phoneme': 'b',
  'confidence': 0.853,
  'support': 4263,
  'coverage': 3638},
 {'pattern': ('a', 'L:n', 'R